# Perturbative Rotations in Atom Interferometry

This notebook demonstrates how rotations perturb a Mach-Zehnder atom
interferometer, following the framework of Ufrecht (Sections 1.3.5, 1.4.2).

**Reference:** C. Ufrecht, *Theoretical approach to high-precision atom
interferometry*, PhD thesis, Universitat Ulm, 2019.

## Two rotation effects

Rotations enter the interferometer through two distinct mechanisms:

1. **Sagnac effect** (Eq. 1.114): A transverse atomic velocity $v_\perp$ in a
   rotating frame produces a Coriolis force $F = 2m\Omega v_\perp$ along the
   laser direction. This acts as a modified gravitational acceleration:
   $$g_{\text{eff}} = g - 2\Omega v_\perp$$
   yielding the Sagnac phase $\Delta\varphi = 2k\Omega v_\perp T^2$.

2. **Phase-space shearing** (xp+px coupling): When the rotation axis has a
   component along the sensitive direction, the Hamiltonian acquires a
   cross-term $\Omega_c(\hat{x}\hat{p}+\hat{p}\hat{x})$. This couples
   position and momentum, shearing the atomic wavepacket in phase space
   and opening the interferometer.

Both effects are treated perturbatively using the BCH expansion.

In [ ]:
import sympy as sy
from sympy import Rational, I, symbols, simplify
from IPython.display import display, Markdown
import numpy as np
from scipy.linalg import expm

from Interferometry import Hamiltonian, Pulse, U, Interferometer
from poly_operator import PolyOpEx, moyal_commutator, bchn, hbar

In [ ]:
# Common parameters
hbar_sym = symbols('hbar')
m_n = Rational(1)
g_n = Rational(1, 100)
k_n = Rational(1, 10)
T_n = Rational(1, 50)

# Baseline: linear gravity, no rotation
H0 = Hamiltonian([Rational(1, 2) / m_n, 0, 0, m_n * g_n, 0, 0])
upper0 = [Pulse(k_n), U(H0, T_n), Pulse(-k_n), U(H0, T_n)]
lower0 = [U(H0, T_n), Pulse(k_n), U(H0, T_n), Pulse(-k_n)]
dic0, _ = Interferometer(upper0, lower0, BCHOrder=8).overlap()

phase0 = complex(simplify(dic0['const']).subs(hbar_sym, 1)).imag
print(f'Baseline MZ phase: {phase0:.10e}')
print(f'Expected -kgT^2:   {-float(k_n * g_n * T_n**2):.10e}')

## 1. Sagnac effect: rotation as modified gravity

For an atom moving with transverse velocity $v_\perp$ in a frame rotating
at rate $\Omega$, the Coriolis force along the laser axis modifies the
effective gravity:

$$g_{\text{eff}} = g - 2\Omega v_\perp$$

The MZ phase becomes $\varphi = -k g_{\text{eff}} T^2$, giving the
Sagnac phase shift:

$$\Delta\varphi_{\text{Sagnac}} = 2k\Omega v_\perp T^2$$

Since this is purely linear gravity with a modified $g$, the interferometer
remains **closed** (no displacement or distortion terms).

In [ ]:
v_perp = Rational(1, 10)  # transverse velocity

omega_vals = [Rational(i, 10000) for i in range(6)]

print('Sagnac phase vs rotation rate')
print(f'{"Omega":>10} {"phi (BCH)":>18} {"phi (expected)":>18} {"Delta_phi":>14}')
print('-' * 65)

for Om in omega_vals:
    g_eff = g_n - 2 * Om * v_perp
    H_sag = Hamiltonian([Rational(1, 2) / m_n, 0, 0, m_n * g_eff, 0, 0])
    up = [Pulse(k_n), U(H_sag, T_n), Pulse(-k_n), U(H_sag, T_n)]
    lo = [U(H_sag, T_n), Pulse(k_n), U(H_sag, T_n), Pulse(-k_n)]
    dic, _ = Interferometer(up, lo, BCHOrder=8).overlap()
    phi = complex(simplify(dic['const']).subs(hbar_sym, 1)).imag
    expected = -float(k_n * g_eff * T_n**2)
    dphi = phi - phase0
    print(f'{float(Om):>10.5f} {phi:>18.12e} {expected:>18.12e} {dphi:>14.6e}')

# Verify Sagnac formula
Om_test = Rational(3, 10000)
g_eff_test = g_n - 2 * Om_test * v_perp
H_test = Hamiltonian([Rational(1, 2) / m_n, 0, 0, m_n * g_eff_test, 0, 0])
up_t = [Pulse(k_n), U(H_test, T_n), Pulse(-k_n), U(H_test, T_n)]
lo_t = [U(H_test, T_n), Pulse(k_n), U(H_test, T_n), Pulse(-k_n)]
dic_t, _ = Interferometer(up_t, lo_t, BCHOrder=8).overlap()
phi_t = complex(simplify(dic_t['const']).subs(hbar_sym, 1)).imag

sagnac_bch = phi_t - phase0
sagnac_formula = 2 * float(k_n * Om_test * v_perp * T_n**2)
print(f'\nSagnac formula verification (Omega={float(Om_test)}):')
print(f'  BCH:     {sagnac_bch:.10e}')
print(f'  Formula: {sagnac_formula:.10e}')
print(f'  Match:   {abs(sagnac_bch - sagnac_formula) < 1e-15}')

In [ ]:
# Verify the interferometer stays closed (pure phase, no displacement)
print('Operator terms for Sagnac (should all be zero):')
for name in ['p2', 'p', 'px_xp', 'x', 'x2']:
    val = abs(complex(simplify(dic_t[name]).subs(hbar_sym, 1)))
    print(f'  |{name}| = {val:.2e}  {"OK" if val < 1e-15 else "NONZERO"}')
print('\nThe interferometer remains closed: pure Sagnac phase, no wavepacket effects.')

## 2. Phase-space shearing: the $\hat{x}\hat{p}+\hat{p}\hat{x}$ perturbation

A more subtle rotation effect arises when the Hamiltonian acquires a
cross-coupling term:

$$H = \frac{\hat{p}^2}{2m} + \Omega_c(\hat{x}\hat{p}+\hat{p}\hat{x}) + mg\hat{x}$$

This corresponds to a rotation that couples position and momentum along
the **same** axis, producing a phase-space shearing.

In pure linear gravity, this perturbation does **not** open the interferometer
at first order in $\Omega_c$ — the displacement terms are $O(\Omega_c^2)$.
The interferometer only opens at second order because the shearing commutes
with the linear evolution at leading order.

In [ ]:
# Scan over several values of the coupling strength
omega_c_vals = [Rational(0), Rational(1, 10000), Rational(2, 10000),
                Rational(5, 10000), Rational(1, 1000)]

print('Overlap coefficients vs phase-space coupling Omega_c')
print(f'{"Omega_c":>10} {"const":>14} {"p coeff":>14} {"x coeff":>14}')
print('-' * 55)

cterm_results = []
for Oc in omega_c_vals:
    H_c = Hamiltonian([Rational(1, 2) / m_n, 0, m_n * Oc, m_n * g_n, 0, 0])
    up = [Pulse(k_n), U(H_c, T_n), Pulse(-k_n), U(H_c, T_n)]
    lo = [U(H_c, T_n), Pulse(k_n), U(H_c, T_n), Pulse(-k_n)]
    dic, opex = Interferometer(up, lo, BCHOrder=8).overlap()
    vals = {n: complex(simplify(dic[n]).subs(hbar_sym, 1)).imag
            for n in ['const', 'p', 'x', 'px_xp', 'p2', 'x2']}
    cterm_results.append((float(Oc), vals, opex))
    print(f'{float(Oc):>10.5f} {vals["const"]:>14.8e} '
          f'{vals["p"]:>14.8e} {vals["x"]:>14.8e}')

In [ ]:
# Extract scaling of displacement with Omega_c
base_vals = cterm_results[0][1]

# The displacement scales as Omega_c^2 (not linearly!) because in linear
# gravity the O(Omega_c) correction keeps the interferometer closed.
# Check d(x) / Omega_c^2 = const.
print('Scaling of displacement with Omega_c:')
print(f'{"Omega_c":>10} {"x coeff":>14} {"x / Oc^2":>14} {"p / Oc^2":>14}')
print('-' * 55)

x_ratios, p_ratios = [], []
for Oc_f, vals, _ in cterm_results[1:]:
    dx = vals['x'] - base_vals['x']
    dp = vals['p'] - base_vals['p']
    x_ratio = dx / Oc_f**2
    p_ratio = dp / Oc_f**2
    x_ratios.append(x_ratio)
    p_ratios.append(p_ratio)
    print(f'{Oc_f:>10.5f} {vals["x"]:>14.6e} {x_ratio:>14.6e} {p_ratio:>14.6e}')

x_spread = (max(x_ratios) - min(x_ratios)) / abs(np.mean(x_ratios))
print(f'\nQuadratic scaling: x/Oc^2 spread = {x_spread:.2e} -> '
      f'{"Quadratic" if x_spread < 0.01 else "Higher order"}')
print('\nThe displacement is O(Omega_c^2): rotation alone does not open')
print('the interferometer at first order in pure linear gravity.')

## 3. Matrix verification

We validate the BCH computation against direct Fock-space matrix
exponentiation for both the pure rotation and the combined
rotation + gravity gradient case.

In [ ]:
# Fock space infrastructure
N_FOCK = 50; M_BLOCK = 20
a_op = np.zeros((N_FOCK, N_FOCK), dtype=complex)
for i in range(N_FOCK - 1):
    a_op[i, i + 1] = np.sqrt(i + 1)
adag_op = a_op.T.copy()
x_mat = (a_op + adag_op) / np.sqrt(2)
p_mat = -1j * (a_op - adag_op) / np.sqrt(2)
I_mat = np.eye(N_FOCK, dtype=complex)

def opex_to_matrix(opex):
    a = complex(simplify(opex.a).subs(hbar_sym, 1))
    b = complex(simplify(opex.b).subs(hbar_sym, 1))
    c = complex(simplify(opex.c).subs(hbar_sym, 1))
    d = complex(simplify(opex.d).subs(hbar_sym, 1))
    e = complex(simplify(opex.e).subs(hbar_sym, 1))
    f = complex(simplify(opex.f).subs(hbar_sym, 1))
    return (a * p_mat @ p_mat + b * p_mat +
            c * (x_mat @ p_mat + p_mat @ x_mat) +
            d * x_mat + e * I_mat + f * x_mat @ x_mat)

def matrix_overlap(H_opex):
    """Compute exact overlap via matrix exponentiation."""
    H_mat = opex_to_matrix(H_opex)
    pulse_p = expm(1j * float(k_n) * x_mat)
    pulse_m = expm(-1j * float(k_n) * x_mat)
    ev = lambda t: expm(-1j * H_mat * float(t))
    U_upper = ev(T_n) @ pulse_m @ ev(T_n) @ pulse_p
    U_lower = pulse_m @ ev(T_n) @ pulse_p @ ev(T_n)
    return U_lower.conj().T @ U_upper

# Test cases
test_cases = [
    ('Pure rotation (Omega_c=0.001)',
     Hamiltonian([Rational(1,2)/m_n, 0, m_n*Rational(1,1000), m_n*g_n, 0, 0])),
    ('Gravity gradient (Gamma=0.001)',
     Hamiltonian([Rational(1,2)/m_n, 0, 0, m_n*g_n, 0, m_n*Rational(1,1000)/2])),
    ('Combined (Omega_c=0.001, Gamma=0.001)',
     Hamiltonian([Rational(1,2)/m_n, 0, m_n*Rational(1,1000),
                  m_n*g_n, 0, m_n*Rational(1,1000)/2])),
]

print('BCH vs matrix exponentiation:')
print('=' * 60)
for label, H_test in test_cases:
    up = [Pulse(k_n), U(H_test, T_n), Pulse(-k_n), U(H_test, T_n)]
    lo = [U(H_test, T_n), Pulse(k_n), U(H_test, T_n), Pulse(-k_n)]
    _, opex_res = Interferometer(up, lo, BCHOrder=8).overlap()

    Z_mat = opex_to_matrix(opex_res)
    expZ = expm(Z_mat)
    overlap_exact = matrix_overlap(H_test)

    err = np.linalg.norm((expZ - overlap_exact)[:M_BLOCK, :M_BLOCK]) / \
          np.linalg.norm(overlap_exact[:M_BLOCK, :M_BLOCK])
    print(f'  {label}: rel_err = {err:.2e}  {"PASS" if err < 1e-6 else "FAIL"}')

## 4. Combined rotation + gravity gradient

In a real experiment, both the gravity gradient $\Gamma_{zz}$ and
the rotation coupling $\Omega_c$ are present simultaneously:

$$H = \frac{\hat{p}^2}{2m} + \Omega_c(\hat{x}\hat{p}+\hat{p}\hat{x}) + mg\hat{x} + \frac{m\Gamma_{zz}}{2}\hat{x}^2$$

We show that at leading order, the two corrections are **additive**:
cross-terms between $\Omega_c$ and $\Gamma_{zz}$ are second-order
and negligible for small perturbations.

In [ ]:
# Check additivity of rotation and gradient corrections
Oc_val = Rational(1, 10000)
gamma_val = Rational(1, 10000)

def overlap_vals(c_coeff, f_coeff):
    H = Hamiltonian([Rational(1, 2) / m_n, 0, c_coeff, m_n * g_n, 0, f_coeff])
    up = [Pulse(k_n), U(H, T_n), Pulse(-k_n), U(H, T_n)]
    lo = [U(H, T_n), Pulse(k_n), U(H, T_n), Pulse(-k_n)]
    dic, _ = Interferometer(up, lo, BCHOrder=8).overlap()
    return {n: complex(simplify(dic[n]).subs(hbar_sym, 1)).imag
            for n in ['const', 'p', 'x']}

r_none = overlap_vals(0, 0)
r_rot  = overlap_vals(m_n * Oc_val, 0)
r_grad = overlap_vals(0, m_n * gamma_val / 2)
r_both = overlap_vals(m_n * Oc_val, m_n * gamma_val / 2)

print('Additivity of perturbations:')
print(f'{"Coeff":>8} {"Rot only":>14} {"Grad only":>14} '
      f'{"Sum":>14} {"Combined":>14} {"Cross-term":>14}')
print('-' * 85)

for name in ['const', 'p', 'x']:
    delta_rot  = r_rot[name]  - r_none[name]
    delta_grad = r_grad[name] - r_none[name]
    sum_parts  = r_none[name] + delta_rot + delta_grad
    combined   = r_both[name]
    cross      = combined - sum_parts
    print(f'{name:>8} {delta_rot:>14.6e} {delta_grad:>14.6e} '
          f'{sum_parts:>14.6e} {combined:>14.6e} {cross:>14.6e}')

print('\nCross-terms are negligible: corrections are additive at leading order.')

In [ ]:
# 2D parameter scan: phase correction as function of (Omega_c, Gamma)
oc_range = [Rational(i, 10000) for i in range(5)]
gam_range = [Rational(i, 10000) for i in range(5)]

print('Phase correction Delta_phi (x10^8) vs (Omega_c, Gamma):')
print(f'{"":>10}', end='')
for g in gam_range:
    print(f'  Gam={float(g):.4f}', end='')
print()
print('-' * 75)

for Oc in oc_range:
    print(f'Oc={float(Oc):.4f}', end='')
    for gamma in gam_range:
        r = overlap_vals(m_n * Oc, m_n * gamma / 2)
        dphi = (r['const'] - r_none['const']) * 1e8
        print(f'{dphi:>13.4f}', end='')
    print()

## 5. Full overlap structure with rotation

We display the complete overlap operator for a combined rotation +
gravity gradient Hamiltonian, showing how the rotation opens the
interferometer through displacement terms in $\hat{p}$ and $\hat{x}$.

In [ ]:
# Full overlap structure for combined perturbation
Oc_show = Rational(1, 1000)
gamma_show = Rational(1, 1000)

H_show = Hamiltonian([Rational(1, 2) / m_n, 0, m_n * Oc_show,
                       m_n * g_n, 0, m_n * gamma_show / 2])
up = [Pulse(k_n), U(H_show, T_n), Pulse(-k_n), U(H_show, T_n)]
lo = [U(H_show, T_n), Pulse(k_n), U(H_show, T_n), Pulse(-k_n)]
dic_show, opex_show = Interferometer(up, lo, BCHOrder=8).overlap()

print(f'Overlap structure (Omega_c={float(Oc_show)}, Gamma={float(gamma_show)})')
print(f'Parameters: k={float(k_n)}, g={float(g_n)}, T={float(T_n)}, m={float(m_n)}, hbar=1')
print('=' * 60)

# Decompose into contributions
r_base = overlap_vals(0, 0)
r_oc   = overlap_vals(m_n * Oc_show, 0)
r_gam  = overlap_vals(0, m_n * gamma_show / 2)
r_full = overlap_vals(m_n * Oc_show, m_n * gamma_show / 2)

print(f'\n{"":>8} {"Baseline":>14} {"+ Rotation":>14} {"+ Gradient":>14} {"Combined":>14}')
print('-' * 60)
for name in ['const', 'p', 'x']:
    print(f'{name:>8} {r_base[name]:>14.6e} {r_oc[name]:>14.6e} '
          f'{r_gam[name]:>14.6e} {r_full[name]:>14.6e}')

# Also show the quadratic terms
print(f'\nQuadratic (distortion) terms:')
for name in ['p2', 'px_xp', 'x2']:
    val = complex(simplify(dic_show[name]).subs(hbar_sym, 1))
    print(f'  {name:8s} = {val.imag:+.6e}i')

# Matrix verification
Z_mat = opex_to_matrix(opex_show)
expZ = expm(Z_mat)
overlap_exact = matrix_overlap(H_show)
err = np.linalg.norm((expZ - overlap_exact)[:M_BLOCK, :M_BLOCK]) / \
      np.linalg.norm(overlap_exact[:M_BLOCK, :M_BLOCK])
print(f'\nMatrix verification: rel_err = {err:.2e}')

## 6. Rotation sensitivity and initial-state dependence

When the interferometer is open (non-zero displacement $\chi$), the
measured phase depends on the initial atomic state $(x_0, p_0)$
through the overlap structure (Eq. 1.108):

$$\varphi_{\text{total}} \approx \text{Im}(e) + \text{Im}(b) \cdot p_0 + \text{Im}(d) \cdot x_0$$

where $b$ and $d$ are the displacement coefficients. This shows how
rotation-induced non-closure creates sensitivity to the initial
momentum (velocity) of the atomic ensemble.

In [ ]:
# Rotation sensitivity: how displacement coefficients scale with Omega_c
Oc_small = Rational(1, 100000)
H_eps = Hamiltonian([Rational(1, 2) / m_n, 0, m_n * Oc_small, m_n * g_n, 0, 0])
up_e = [Pulse(k_n), U(H_eps, T_n), Pulse(-k_n), U(H_eps, T_n)]
lo_e = [U(H_eps, T_n), Pulse(k_n), U(H_eps, T_n), Pulse(-k_n)]
dic_eps, _ = Interferometer(up_e, lo_e, BCHOrder=8).overlap()

# Slopes d(coeff)/d(Omega_c)
print('Rotation displacement sensitivity (d/d(Omega_c)):')
for name in ['const', 'p', 'x']:
    v0 = complex(simplify(dic0[name]).subs(hbar_sym, 1)).imag
    v1 = complex(simplify(dic_eps[name]).subs(hbar_sym, 1)).imag
    slope = (v1 - v0) / float(Oc_small)
    print(f'  d({name}.imag)/d(Omega_c) = {slope:.8e}')

# The position displacement tells us the momentum kick from rotation
dx_slope = complex(simplify(dic_eps['x']).subs(hbar_sym, 1)).imag / float(Oc_small)
dp_slope = complex(simplify(dic_eps['p']).subs(hbar_sym, 1)).imag / float(Oc_small)

print(f'\nFor an atom with initial momentum p_0, the rotation adds to the phase:')
print(f'  delta_phi(p_0) = {dp_slope:.4e} * Omega_c * p_0')
print(f'  delta_phi(x_0) = {dx_slope:.4e} * Omega_c * x_0')
print(f'\nThis initial-state dependence is the signature of an open interferometer.')
print(f'It leads to contrast loss when averaging over a thermal ensemble.')

## Summary

This notebook demonstrated two rotation effects in atom interferometry:

1. **Sagnac effect**: Transverse velocity in a rotating frame modifies the
   effective gravity, giving $\Delta\varphi = 2k\Omega v_\perp T^2$.
   The interferometer remains closed (pure phase).

2. **Phase-space shearing** ($\hat{x}\hat{p}+\hat{p}\hat{x}$ coupling):
   Opens the interferometer with displacement terms scaling linearly
   in $\Omega_c$. Creates initial-state-dependent phase shifts.

3. **Combined perturbations**: Rotation and gravity gradient corrections
   are additive at leading order, with negligible cross-terms.

All results are validated against direct Fock-space matrix exponentiation.